# Last-iterate figures for S04 (Learning algorithms II)

This notebook simulates the learning dynamics shown in the last-iterate figures of `content/learning2.typ` and writes the data files read by the figure sources in `content/figures/learning2/`:

| Data file | Figure source |
|---|---|
| `last_vs_average.json` | `last_vs_average.typ` (average versus last iterate) |
| `comparison.json` | `comparison_trajectories.typ`, `comparison_gaps.typ` (four optimistic algorithms) |
| `forgetful.json` | `forgetful.typ` (plateau and rebound of optimistic MWU) |

Only the Python standard library is needed (the optional precision check uses `mpmath`). After rerunning, recompile the figures, for example with `make figures`.

**Conventions.** All games are two-player zero-sum $2\times 2$ games in which Player 1 maximizes $x^\top U y$ and Player 2 maximizes $-x^\top U y$, and both players start from the uniform strategy. The saddle-point gap is $\gamma(x, y) = \max_i (Uy)_i - \min_j (x^\top U)_j$. For two actions, the entropic updates are computed from the log-odds of the cumulative utilities, which keeps probabilities as small as $10^{-300}$ accurate in double precision. With $\delta = 0.01$ and $\eta = 0.1$, this reproduces the optimistic MWU iterate at $t = 30000$ shown in the lecture slides: $x_2 = 9.23\cdot 10^{-12}$, $y_1 = 0.180$, gap $8.20\cdot 10^{-3}$.

In [ ]:
import json
import math
from pathlib import Path

# Locate the repository root from the notebook's working directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'content/learning2.typ').is_file())
OUTPUT = ROOT / 'content/figures/learning2'
FLOOR = 1e-12  # Plot floor; values below it are not drawn.
BINS_PER_DECADE = 40

## Learning updates for two actions

In [ ]:
def logistic(z):
    """Return (p, 1 - p) for p = 1 / (1 + exp(-z)) without overflow."""
    if z >= 0:
        e = math.exp(-z)
        return (1 / (1 + e), e / (1 + e))
    e = math.exp(z)
    return (e / (1 + e), 1 / (1 + e))


def euclidean_choice(z):
    """argmax_x <theta, x> - ||x||^2 / (2 eta) on the 2-simplex, as a function
    of z = eta (theta_1 - theta_2): the projection of eta theta."""
    p = min(1.0, max(0.0, 0.5 + z / 2))
    return (p, 1 - p)


def log_choice(z):
    """argmax_x <theta, x> + (1 / eta) sum_i log x_i on the 2-simplex, as a
    function of z = eta (theta_1 - theta_2). The optimality conditions give
    x_i = 1 / (eta (lambda - theta_i)); with a = eta (lambda - theta_1),
    1/a + 1/(a + z) = 1, whose positive root is used below."""
    a = ((2 - z) + math.sqrt(4 + z * z)) / 2
    p = 1 / a
    return (p, 1 - p)


CHOICE = {'entropy': logistic, 'euclidean': euclidean_choice, 'log': log_choice}


def project(v):
    """Euclidean projection of a 2-vector onto the simplex."""
    p = min(1.0, max(0.0, 0.5 + (v[0] - v[1]) / 2))
    return (p, 1 - p)


def feedback(U, x, y):
    """Utility gradients of both players and the equilibrium gap at (x, y)."""
    Uy = [U[0][0] * y[0] + U[0][1] * y[1], U[1][0] * y[0] + U[1][1] * y[1]]
    xU = [x[0] * U[0][0] + x[1] * U[1][0], x[0] * U[0][1] + x[1] * U[1][1]]
    return Uy, [-v for v in xU], max(Uy) - min(xU)


def gap(U, x, y):
    return feedback(U, x, y)[2]

## Simulators

In [ ]:
def simulate(U, algorithm, eta, T):
    """Return the last-iterate and average-iterate gaps for t = 1, ..., T.

    algorithm is one of
      'mwu', 'omwu'  : (optimistic) MWU = (optimistic) FTRL with entropy;
      'ogd', 'ogda'  : (optimistic) projected gradient ascent, the optimistic
                       variant being the non-reflected update of the notes:
                       z' = P(z + eta g), x' = P(z' + eta g).
    """
    last, average = [], []
    ax, ay = [0.0, 0.0], [0.0, 0.0]
    if algorithm in ('mwu', 'omwu'):
        Gx = Gy = 0.0  # Cumulative utility difference between the two actions.
        px = py = 0.0  # Last utility difference (the optimistic prediction).
        for t in range(1, T + 1):
            if algorithm == 'mwu':
                x, y = logistic(eta * Gx), logistic(eta * Gy)
            else:
                x, y = logistic(eta * (Gx + px)), logistic(eta * (Gy + py))
            gx, gy, g = feedback(U, x, y)
            ax = [a + (b - a) / t for a, b in zip(ax, x)]
            ay = [a + (b - a) / t for a, b in zip(ay, y)]
            last.append(g)
            average.append(gap(U, ax, ay))
            px, py = gx[0] - gx[1], gy[0] - gy[1]
            Gx, Gy = Gx + px, Gy + py
    else:
        x = zx = (0.5, 0.5)
        y = zy = (0.5, 0.5)
        for t in range(1, T + 1):
            gx, gy, g = feedback(U, x, y)
            ax = [a + (b - a) / t for a, b in zip(ax, x)]
            ay = [a + (b - a) / t for a, b in zip(ay, y)]
            last.append(g)
            average.append(gap(U, ax, ay))
            if algorithm == 'ogd':
                x = project([x[i] + eta * gx[i] for i in range(2)])
                y = project([y[i] + eta * gy[i] for i in range(2)])
            else:
                zx = project([zx[i] + eta * gx[i] for i in range(2)])
                zy = project([zy[i] + eta * gy[i] for i in range(2)])
                x = project([zx[i] + eta * gx[i] for i in range(2)])
                y = project([zy[i] + eta * gy[i] for i in range(2)])
    return last, average


def simulate_oftrl(U, regularizer, eta, T):
    """Optimistic FTRL with prediction m^(t+1) = g^(t), for both players.

    Returns the strategies x^(t), y^(t) and the gaps for t = 1, ..., T.
    The entropic case is optimistic MWU, computed in log-odds form."""
    choice = CHOICE[regularizer]
    Gx = Gy = px = py = 0.0
    xs, ys, gaps = [], [], []
    for _ in range(T):
        x = choice(eta * (Gx + px))
        y = choice(eta * (Gy + py))
        gx, gy, g = feedback(U, x, y)
        xs.append(x)
        ys.append(y)
        gaps.append(g)
        px, py = gx[0] - gx[1], gy[0] - gy[1]
        Gx, Gy = Gx + px, Gy + py
    return xs, ys, gaps


def simulate_ogda(U, eta, T):
    """Non-reflected optimistic projected gradient ascent (the notes' (1))."""
    x = zx = (0.5, 0.5)
    y = zy = (0.5, 0.5)
    xs, ys, gaps = [], [], []
    for _ in range(T):
        gx, gy, g = feedback(U, x, y)
        xs.append(x)
        ys.append(y)
        gaps.append(g)
        zx = project([zx[i] + eta * gx[i] for i in range(2)])
        zy = project([zy[i] + eta * gy[i] for i in range(2)])
        x = project([zx[i] + eta * gx[i] for i in range(2)])
        y = project([zy[i] + eta * gy[i] for i in range(2)])
    return xs, ys, gaps

## Plotting helpers

`plateau_times` uses the times $T_1, T_2$ of the analysis of Cai et al. (2024). `envelope` summarizes a sequence by its minimum and maximum over logarithmically spaced windows, so that oscillations are not aliased by downsampling.

In [ ]:
def plateau_times(xs, ys, delta):
    """The times of the analysis of Cai et al. (2024): T1 is the first time
    at which x_1 >= 1 / (1 + delta), and T2 the first time after T1 at which
    y_1 >= 1 / (2 (1 + delta)); both thresholds are equilibrium probabilities.
    Either is None if it does not occur within the horizon."""
    t1 = next((t for t in range(1, len(xs) + 1)
               if xs[t - 1][0] >= 1 / (1 + delta)), None)
    if t1 is None:
        return None, None
    t2 = next((t for t in range(t1 + 1, len(ys) + 1)
               if ys[t - 1][0] >= 1 / (2 * (1 + delta))), None)
    return t1, t2


def thin(points, tolerance=0.006):
    """Keep a trajectory point whenever it moved by at least `tolerance`."""
    kept = [points[0]]
    for p in points[1:]:
        if math.dist(p, kept[-1]) >= tolerance:
            kept.append(p)
    if kept[-1] != points[-1]:
        kept.append(points[-1])
    return [[round(a, 5), round(b, 5)] for a, b in kept]


def envelope(values, floor=FLOOR, clip=False):
    """Per logarithmic bin of t: [log10 t, log10 min, log10 max].

    Oscillating sequences are drawn as a band between the per-bin minimum and
    maximum so that downsampling cannot alias them. Minima are clipped to the
    floor. Unless `clip` is set, the series ends at the first bin whose maximum
    lies below the floor; with `clip`, such bins are kept at the floor.
    """
    T = len(values)
    edges = sorted({min(T, round(10 ** (k / BINS_PER_DECADE)))
                    for k in range(int(BINS_PER_DECADE * math.log10(T)) + 2)})
    rows, start = [], 1
    for end in edges:
        if end < start:
            continue
        chunk = values[start - 1:end]
        start = end + 1
        hi, lo = max(chunk), min(chunk)
        if hi < floor and not clip:
            break
        t = math.sqrt((end - len(chunk) + 1) * end)  # Geometric bin center.
        rows.append([round(math.log10(t), 4),
                     round(math.log10(max(lo, floor)), 4),
                     round(math.log10(max(hi, floor)), 4)])
    return rows

## Average versus last iterate (`last_vs_average.json`)

In [ ]:
# Running example of the notes: same game, learning rates, and starting point
# as the trajectory figure of Section S4.1.1.
U = [[2, 1], [0, 2]]
T = 10_000
panels = []
for name, (plain, optimistic, eta) in {
    'MWU': ('mwu', 'omwu', 0.25),
    'OGD': ('ogd', 'ogda', 0.1),
}.items():
    panel = {'name': name, 'eta': eta}
    for key, algorithm in (('plain', plain), ('optimistic', optimistic)):
        last, average = simulate(U, algorithm, eta, T)
        panel[key] = {'last': envelope(last), 'average': envelope(average)}
    panels.append(panel)
(OUTPUT / 'last_vs_average.json').write_text(
    json.dumps({'T': T, 'panels': panels}, separators=(',', ':')) + '\n')

## Plateau and rebound of optimistic MWU for three values of $\delta$ (`forgetful.json`)

In [ ]:
# Hard game of Cai et al. (2024), written as a utility matrix for Player 1.
T = 20_000
eta = 0.1
panels = []
for delta in (0.05, 0.01, 0.005):
    U = [[0.5 - delta, 0.5], [1.0, 0.0]]
    xs, ys, omwu = simulate_oftrl(U, 'entropy', eta, T)
    ogda, _ = simulate(U, 'ogda', eta, T)
    t1, t2 = plateau_times(xs, ys, delta)
    panels.append({
        'delta': delta, 't1': t1, 't2': t2,
        'gap1': round(math.log10(omwu[t1 - 1]), 4),
        'gap2': round(math.log10(omwu[t2 - 1]), 4),
        'omwu': envelope(omwu), 'ogda': envelope(ogda),
    })
    print(f'delta={delta}: T1={t1}, T2={t2}, '
          f'OMWU gap at T: {omwu[-1]:.3e}, OGDA gap at T: {ogda[-1]:.3e}')
(OUTPUT / 'forgetful.json').write_text(
    json.dumps({'T': T, 'eta': eta, 'panels': panels}, separators=(',', ':')) + '\n')

## Four optimistic algorithms in the game with $\delta = 0.01$ (`comparison.json`)

In [ ]:
# The same game with delta = 0.01, for three optimistic FTRL variants and
# optimistic OGD (compare Cai et al., 2024, Figure 1).
delta, T, eta = 0.01, 100_000, 0.1
U = [[0.5 - delta, 0.5], [1.0, 0.0]]
algorithms = []
for name, regularizer in (('Entropy (optimistic MWU)', 'entropy'),
                          ('Squared Euclidean norm', 'euclidean'),
                          ('Logarithmic regularizer', 'log'),
                          ('Optimistic OGD', None)):
    if regularizer is None:
        xs, ys, gaps = simulate_ogda(U, eta, T)
    else:
        xs, ys, gaps = simulate_oftrl(U, regularizer, eta, T)
    entry = {
        'name': name,
        'trajectory': thin([(x[0], y[0]) for x, y in zip(xs, ys)]),
        'gap': envelope(gaps),
        # Player 1's probability of its second action, clipped at 1e-40
        # (so that the exact zeros of the projection-based methods show).
        'x2': envelope([x[1] for x in xs], floor=1e-40, clip=True),
        'x2_min': min(x[1] for x in xs),
    }
    if regularizer is not None:
        t1, t2 = plateau_times(xs, ys, delta)
        for key, t in (('1', t1), ('2', t2)):
            if t is not None:
                entry['t' + key] = t
                entry['p' + key] = [round(xs[t - 1][0], 5), round(ys[t - 1][0], 5)]
                entry['gap' + key] = round(math.log10(gaps[t - 1]), 4)
    algorithms.append(entry)
    print(f"{name}: min x2 = {entry['x2_min']:.3e}, "
          f"T1 = {entry.get('t1')}, T2 = {entry.get('t2')}, final gap = {gaps[-1]:.3e}")
(OUTPUT / 'comparison.json').write_text(json.dumps(
    {'T': T, 'eta': eta, 'delta': delta,
     'equilibrium': [1 / (1 + delta), 1 / (2 * (1 + delta))],
     'algorithms': algorithms}, separators=(',', ':')) + '\n')

## Optional: precision check

In [ ]:
# Optional: check the double-precision optimistic FTRL runs against 120-digit
# arithmetic, using the multiplicative (exponential) form for the entropy.
# Requires mpmath (pip install mpmath); takes about half a minute.
try:
    from mpmath import mp, mpf, exp, sqrt
except ImportError:
    print('mpmath is not installed; skipping the precision check.')
else:
    mp.dps = 120

    def oftrl_high_precision(regularizer, delta, eta, T):
        d, e = mpf(delta), mpf(eta)
        U = [[mpf(1) / 2 - d, mpf(1) / 2], [mpf(1), mpf(0)]]

        def choice(z):
            if regularizer == 'entropy':
                a, b = exp(z / 2), exp(-z / 2)
                return (a / (a + b), b / (a + b))
            a = ((2 - z) + sqrt(4 + z * z)) / 2
            return (1 / a, 1 - 1 / a)

        Gx = Gy = px = py = mpf(0)
        gaps = []
        for _ in range(T):
            x, y = choice(e * (Gx + px)), choice(e * (Gy + py))
            Uy = [U[0][0] * y[0] + U[0][1] * y[1], U[1][0] * y[0] + U[1][1] * y[1]]
            xU = [x[0] * U[0][0] + x[1] * U[1][0], x[0] * U[0][1] + x[1] * U[1][1]]
            gaps.append(max(Uy) - min(xU))
            px, py = Uy[0] - Uy[1], -(xU[0] - xU[1])
            Gx, Gy = Gx + px, Gy + py
        return gaps

    runs = [('entropy', delta, 20_000) for delta in (0.05, 0.01, 0.005)]
    runs += [('entropy', 0.01, 100_000), ('log', 0.01, 100_000)]
    for regularizer, delta, T in runs:
        exact = oftrl_high_precision(regularizer, delta, 0.1, T)
        _, _, double = simulate_oftrl([[0.5 - delta, 0.5], [1.0, 0.0]], regularizer, 0.1, T)
        error = max(abs(float(a) - b) / float(a) for a, b in zip(exact, double))
        print(f'{regularizer}, delta={delta}, T={T}: max relative gap error {error:.1e}')